Importing the dependecies


In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score



In [2]:
# loading dataset to a pandas data frame
credit_card_data = pd.read_csv('/content/credit_card_fraud_10k.csv')

In [3]:
#first five rpows of the data set
credit_card_data.head()

,transaction_id,amount,transaction_hour,merchant_category,foreign_transaction,location_mismatch,device_trust_score,velocity_last_24h,cardholder_age,is_fraud
0,1,84.47,22,Electronics,0,0,66,3,40,0
1,2,541.82,3,Travel,1,0,87,1,64,0
2,3,237.01,17,Grocery,0,0,49,1,61,0
3,4,164.33,4,Grocery,0,1,72,3,34,0
4,5,30.53,15,Food,0,0,79,0,44,0


In [5]:
credit_card_data.tail()

,transaction_id,amount,transaction_hour,merchant_category,foreign_transaction,location_mismatch,device_trust_score,velocity_last_24h,cardholder_age,is_fraud
9995,9996,350.91,22,Food,0,0,99,4,37,0
9996,9997,410.04,5,Clothing,0,0,70,3,25,0
9997,9998,527.75,21,Electronics,0,0,44,2,45,0
9998,9999,91.20,2,Electronics,0,0,38,0,37,0
9999,10000,44.06,2,Clothing,0,0,38,0,66,0


In [6]:
#dataset information
credit_card_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 10 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   transaction_id       10000 non-null  int64  
 1   amount               10000 non-null  float64
 2   transaction_hour     10000 non-null  int64  
 3   merchant_category    10000 non-null  object 
 4   foreign_transaction  10000 non-null  int64  
 5   location_mismatch    10000 non-null  int64  
 6   device_trust_score   10000 non-null  int64  
 7   velocity_last_24h    10000 non-null  int64  
 8   cardholder_age       10000 non-null  int64  
 9   is_fraud             10000 non-null  int64  
dtypes: float64(1), int64(8), object(1)
memory usage: 781.4+ KB


In [7]:
#checking the number of missing values in each column
credit_card_data.isnull().sum()

,0
transaction_id,0
amount,0
transaction_hour,0
merchant_category,0
foreign_transaction,0
location_mismatch,0
device_trust_score,0
velocity_last_24h,0
cardholder_age,0
is_fraud,0


In [10]:
#check the distribution of legit transaction and fraudulent transaction
credit_card_data['is_fraud'].value_counts()

,count
is_fraud,
0,9849
1,151


this dataset is highly unbalanced

0---> normal transaction
1---> fraudulent transaction

In [11]:
#seperating the data for analysis
legit = credit_card_data[credit_card_data.is_fraud == 0]
fraud = credit_card_data[credit_card_data.is_fraud == 1]

In [12]:
print(legit.shape)
print(fraud.shape)

(9849, 10)
(151, 10)


In [13]:
#statistical measures for data
legit.amount.describe()

,amount
count,9849.000000
mean,175.333015
std,173.986837
min,0.000000
25%,50.990000
50%,122.110000
75%,241.650000
max,1471.040000


In [15]:
fraud.amount.describe()

,amount
count,151.000000
mean,216.182980
std,248.120467
min,0.110000
25%,41.530000
50%,118.940000
75%,341.695000
max,1185.070000


In [17]:
#compare the values for both transactions
credit_card_data.groupby('is_fraud').mean(numeric_only=True)

,transaction_id,amount,transaction_hour,foreign_transaction,location_mismatch,device_trust_score,velocity_last_24h,cardholder_age
is_fraud,,,,,,,,
0,5004.129861,175.333015,11.712154,0.090974,0.079704,62.165804,1.990557,43.469794
1,4763.741722,216.182980,3.841060,0.543046,0.476821,37.867550,3.205298,43.397351


Under sampling
build a sample dataset containing similar distribution of normal transactions and fraudulent transaction

Number of fraudulent transaction is 151

In [18]:
legit_sample = legit.sample(n=151)


Concatenating two dataframes

In [19]:
new_dataset = pd.concat([legit_sample, fraud],axis=0)

In [20]:
new_dataset.head()

,transaction_id,amount,transaction_hour,merchant_category,foreign_transaction,location_mismatch,device_trust_score,velocity_last_24h,cardholder_age,is_fraud
6569,6570,98.13,17,Travel,0,0,27,3,42,0
8205,8206,53.01,22,Clothing,0,0,97,1,62,0
7821,7822,86.38,12,Clothing,1,0,56,4,26,0
8772,8773,146.24,13,Travel,0,0,78,0,61,0
8999,9000,483.74,8,Travel,0,0,50,4,68,0


In [21]:
new_dataset.tail()

,transaction_id,amount,transaction_hour,merchant_category,foreign_transaction,location_mismatch,device_trust_score,velocity_last_24h,cardholder_age,is_fraud
9526,9527,398.66,3,Grocery,0,0,30,5,61,1
9553,9554,182.65,2,Electronics,1,1,31,2,28,1
9587,9588,93.93,1,Travel,0,1,27,3,69,1
9779,9780,172.00,2,Grocery,0,1,56,5,24,1
9973,9974,11.22,1,Grocery,0,1,25,4,32,1


In [22]:
new_dataset['is_fraud'].value_counts()

,count
is_fraud,
0,151
1,151


In [23]:
new_dataset.groupby('is_fraud').mean(numeric_only=True)

,transaction_id,amount,transaction_hour,foreign_transaction,location_mismatch,device_trust_score,velocity_last_24h,cardholder_age
is_fraud,,,,,,,,
0,5431.682119,185.86755,11.450331,0.099338,0.066225,62.039735,2.192053,43.980132
1,4763.741722,216.18298,3.841060,0.543046,0.476821,37.867550,3.205298,43.397351


Splitting the data into features and targets

In [24]:
X = new_dataset.drop(columns='is_fraud', axis = 1)
Y = new_dataset['is_fraud']

In [25]:
print(X)

      transaction_id  amount  ...  velocity_last_24h cardholder_age
6569            6570   98.13  ...                  3             42
8205            8206   53.01  ...                  1             62
7821            7822   86.38  ...                  4             26
8772            8773  146.24  ...                  0             61
8999            9000  483.74  ...                  4             68
...              ...     ...  ...                ...            ...
9526            9527  398.66  ...                  5             61
9553            9554  182.65  ...                  2             28
9587            9588   93.93  ...                  3             69
9779            9780  172.00  ...                  5             24
9973            9974   11.22  ...                  4             32

[302 rows x 9 columns]


In [26]:
print(Y)

6569    0
8205    0
7821    0
8772    0
8999    0
       ..
9526    1
9553    1
9587    1
9779    1
9973    1
Name: is_fraud, Length: 302, dtype: int64


To split the data into training data and testing data

In [27]:
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, stratify=Y, random_state=2)

In [28]:
print(X.shape, X_train.shape, X_test.shape)

(302, 9) (241, 9) (61, 9)


Model training

Logistic regression

In [35]:
model = LogisticRegression(max_iter=1000)

In [31]:
# drop non-numeric columns and id columns for training
X_numeric = X.drop(columns=['transaction_id', 'merchant_category'], errors='ignore')

# split the clean numeric data again
X_train_num, X_test_num, Y_train_num, Y_test_num = train_test_split(X_numeric, Y, test_size=0.2, stratify=Y, random_state=2)

# training the logistic regression model
model.fit(X_train_num, Y_train_num)

/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


LogisticRegression()

Evaluation

Accuracy score

In [38]:
# refit the model to ensure it is trained before predicting
model.fit(X_train_num, Y_train_num)

# accuracy on training data
X_train_prediction = model.predict(X_train_num)
training_data_accuracy = accuracy_score(X_train_prediction, Y_train_num)
print('Accuracy score on training data:', training_data_accuracy)

Accuracy score on training data: 0.970954356846473
